# ADF

A number of plots are provided from ADF. The full output from the stand-alone ADF configuration is in the link below.


Note that in standalone format (eg, CUPiD run not through CESM workflow), ADF is currently run by users via the following process:
1) Install ADF and activate cupid-analysis
2) Use the `CUPiD/helper_scripts/generate_adf_config_file.py` script to generate an ADF config file based on a CUPiD configuration file.
   * `cd CUPiD/examples/external_diag_packages`
   * `../../helper_scripts/generate_adf_config_file.py --cupid-config-loc . --adf-template ../../externals/ADF/config_amwg_default_plots.yaml --out-file ADF_config.yaml`
3) Run ADF with the newly created configuration file.
   * `../../externals/ADF/run_adf_diag ADF_config.yaml`

In [ ]:
import os

from IPython.core.display import HTML, Image
from IPython.display import display
import pandas as pd

In [ ]:
adf_root = "."
case_names = []
start_dates = []
end_dates = []
key_plots = None
compare_obs: False
# adf_root will be external_diag_packages/computed_notebooks/ADF/

In [ ]:
# Want some base case parameter defaults to equal control case values
if compare_obs:
    base_case_name = "Obs"
    base_start_date = None
    base_end_date = None

In [ ]:
# convert start-date and end-date to year rang

base_case_name = case_names[0]
base_start_date = start_dates[0]
base_end_date = end_dates[0]

base_case_year_range = [
    int(base_start_date.split("-")[0]),
    int(base_end_date.split("-")[0]),
]

case_info = {}
for case_name, start_date, end_date in zip(
    case_names[1:], start_dates[1:], end_dates[1:]
):
    case_info[case_name] = {
        "start_date": start_date,
        "end_date": end_date,
        "case_year_range": [int(start_date.split("-")[0]), int(end_date.split("-")[0])],
    }

In [ ]:
if base_case_year_range:
    base_case_yr_range_str = f"_{base_case_year_range[0]}_{base_case_year_range[1]}"
    alt_base_case_yr_range_str = (
        f"_{base_case_year_range[0]}_{str(int(base_case_year_range[1])-1)}"
    )
else:
    base_case_yr_range_str = ""
    alt_base_case_yr_range_str = ""

for case_name, info in case_info.items():
    case_year_range = info["case_year_range"]
    case_year_range_str = f"_{case_year_range[0]}_{case_year_range[1]}"
    alternate_case_year_range_str = (
        f"_{case_year_range[0]}_{str(int(case_year_range[1])-1)}"
    )

    possible_adf_comparison_names = [
        f"{base_case_name}{base_case_yr_range_str}_vs_{case_name}{case_year_range_str}"
    ]
    possible_adf_comparison_names.append(
        f"{base_case_name}{alt_base_case_yr_range_str}_vs_{case_name}{case_year_range_str}"
    )
    possible_adf_comparison_names.append(
        f"{base_case_name}{alt_base_case_yr_range_str}_vs_{case_name}{alternate_case_year_range_str}"
    )
    possible_adf_comparison_names.append(
        f"{base_case_name}{base_case_yr_range_str}_vs_{case_name}{alternate_case_year_range_str}"
    )

    case_info[case_name][
        "possible_adf_comparison_names"
    ] = possible_adf_comparison_names

In [ ]:
for case_name, info in case_info.items():
    possible_adf_comparison_names = info["possible_adf_comparison_names"]
    adf_found = False

    for adf_comparison_name in possible_adf_comparison_names:
        adf_root_candidate = os.path.join(adf_root, adf_comparison_name)
        if os.path.exists(adf_root_candidate):
            adf_found = True
            matched_adf_root = adf_root_candidate

            display(
                HTML(
                    f'<a href="./ADF/{adf_comparison_name}/website/index.html" target="_blank" style="font-size: 30px">Full ADF output for base case vs{case_name}</a>'
                )
            )
            case_info[case_name]["matched_adf_root"] = matched_adf_root
            break

    if not adf_found:
        matched_adf_root = None
        case_info[case_name]["matched_adf_root"] = None
        print(
            f"{case_name}: No ADF output found for the specified case and date range."
        )

## Key Metrics from ADF

Some important things to look at from ADF include a comparison table and a few maps:

In [ ]:
for case_name, info in case_info.items():
    adf_root = info["matched_adf_root"]
    if adf_root is None:
        continue
    else:
        comparison_table = os.path.join(adf_root, "amwg_table_comp.csv")
        if os.path.isfile(comparison_table):
            table = pd.read_csv(comparison_table)
            display(HTML(table.to_html(index=False, float_format="{:6g}".format)))

In [ ]:
for case_name, info in case_info.items():
    adf_root = info["matched_adf_root"]
    if adf_root is None:
        print(f"{case_name}: No matched ADF root found, skipping key plots.")
        continue
    else:
        for path_to_key_plot in key_plots:
            full_path = os.path.join(adf_root, path_to_key_plot)
            print("full_path:", full_path)
            if os.path.isfile(full_path):
                display(Image(full_path))